# KinOpt educational workflow: local kinase-effect fitting

**Purpose.** KinOpt estimates how upstream kinase activity explains observed phosphorylation-site dynamics. This notebook builds a tiny kinase–substrate example and walks through the real KinOpt preprocessing and objective-evaluation utilities, then performs a lightweight JAXopt local optimization to form a **ranked multistart solution ensemble**.

Scientifically, each substrate phosphorylation site is modeled as a weighted mixture of candidate kinase effects. Computationally, the workflow converts biological tables into fixed arrays, solves a **single-objective constrained optimization** problem, ranks starts by scalar loss, and exports parameters and plots.

## Mathematical problem

For substrate site \(i\), kinase \(k\), kinase phosphorylation/readout row \(r\), and time \(t\):

$$
M_k(t) = \sum_{r \in \mathcal{R}(k)} \beta_{kr} K_{r}(t), \qquad
\hat P_i(t) = \sum_{k \in \mathcal{K}(i)} \alpha_{ik} M_k(t)
$$

The scalar objective is mean squared error:

$$
L(\alpha,\beta)=\frac{1}{n}\sum_i\sum_t(P_i(t)-\hat P_i(t))^2.
$$

Here \(\alpha\) distributes influence among candidate kinases for each substrate site and \(\beta\) combines kinase-level measured rows. We enforce simplex-style constraints in the notebook optimizer so each \(\alpha\) block and each \(\beta\) block sums to one. The fitted result is interpreted as a local optimum of a single scalar loss, not as a multi-objective front.

In [ ]:

from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "README.md").exists():
    REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT.parent))
import os
os.chdir(REPO_ROOT)

FAST_NOTEBOOK = True
RANDOM_SEED = 7

import json
from types import SimpleNamespace

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
import jaxopt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from kinopt.local.optcon.construct import (
    _build_P_initial, _build_K_data, _convert_to_sparse, _precompute_mappings,
    _compute_time_weights,
)
from kinopt.local.objfn.minfn import _objective
from kinopt.local.objfn import estimated_series
from kinopt.local.utils.params import extract_parameters, compute_metrics
from kinopt.local.exporter.plotout import export_outcomes_to_csv, plot_multistart_summary_runtime_overlay
from networkmodel.backend import project_alpha_blocks, project_beta_blocks

np.random.seed(RANDOM_SEED)
print("JAX float64 enabled:", jax.config.jax_enable_x64)


## Dummy KinOpt input tables

KinOpt expects a phosphorylation table with `GeneID`, `Psite`, and time columns (`x1` … `x14`) plus an interaction table mapping each substrate site to candidate kinases. The synthetic data below has two substrate sites and two kinases. Fourteen columns are supplied because the current KinOpt preprocessing utilities expect that schema, even though this educational example uses only tiny smooth curves.

In [ ]:

time_cols = [f"x{i}" for i in range(1, 15)]
t = np.arange(14, dtype=float)

def curve(base, slope, wave=0.0):
    return base + slope * t + wave * np.sin(t / 2)

full_df = pd.DataFrame([
    {"GeneID": "SUB1", "Psite": "S10", **dict(zip(time_cols, curve(1.00, 0.035, 0.03)))},
    {"GeneID": "SUB2", "Psite": "T22", **dict(zip(time_cols, curve(0.90, 0.020, -0.02)))},
    {"GeneID": "KIN_A", "Psite": "Y100", **dict(zip(time_cols, curve(1.10, 0.040, 0.02)))},
    {"GeneID": "KIN_B", "Psite": "S200", **dict(zip(time_cols, curve(0.85, 0.015, 0.04)))},
])
interact_df = pd.DataFrame([
    {"GeneID": "SUB1", "Psite": "S10", "Kinase": ["KIN_A", "KIN_B"]},
    {"GeneID": "SUB2", "Psite": "T22", "Kinase": ["KIN_B"]},
])
display(full_df.head())
display(interact_df)


## Preprocess into KinOpt arrays

The package preprocessing maps biological identifiers into compact numeric arrays: observed substrate matrix `P_array`, kinase readout matrix `K_array`, sparse kinase storage, and index vectors that locate each parameter block. This step is necessary because repeated objective calls must be fast and shape-stable.

In [ ]:

P_initial, P_array = _build_P_initial(full_df, interact_df)
K_index, K_array, beta_counts = _build_K_data(full_df, interact_df, estimate_missing=False)
K_sparse, K_data, K_indices, K_indptr = _convert_to_sparse(K_array)
(unique_kinases, gene_kinase_counts, gene_alpha_starts, gene_kinase_idx,
 total_alpha, kinase_beta_counts, kinase_beta_starts) = _precompute_mappings(P_initial, K_index)
t_max, P_dense, time_weights = _compute_time_weights(P_array, loss_type="mse")

print("unique_kinases:", unique_kinases)
print("P_array shape:", P_array.shape, "K_array shape:", K_array.shape)
print("total alpha parameters:", total_alpha, "total beta parameters:", int(sum(kinase_beta_counts)))
param_layout = pd.DataFrame({
    "substrate_site": [str(k) for k in P_initial.keys()],
    "candidate_kinases": [", ".join(v["Kinases"]) for v in P_initial.values()],
    "n_alpha": gene_kinase_counts,
})
display(param_layout)


## Model setup and local optimization

We evaluate the real KinOpt numba objective for diagnostics. For the educational local optimizer we use an equivalent JAX objective and JAXopt projected-gradient solver so this notebook follows the current constrained-local-optimization style used by the active JAX/JAXopt modules. Projection keeps each biological weight block interpretable as a convex contribution.

In [ ]:

# Block identifiers for simplex projection.
alpha_block_ids = np.concatenate([np.full(c, i) for i, c in enumerate(gene_kinase_counts)])
beta_block_ids = np.concatenate([np.full(c, i) for i, c in enumerate(kinase_beta_counts)])
n_beta = len(beta_block_ids)

P_j = jnp.asarray(P_dense)
K_j = jnp.asarray(K_array)
gene_starts_j = jnp.asarray(gene_alpha_starts)
gene_counts_j = jnp.asarray(gene_kinase_counts)
gene_kinase_idx_j = jnp.asarray(gene_kinase_idx)
kin_beta_starts_j = jnp.asarray(kinase_beta_starts)
kin_beta_counts_j = jnp.asarray(kinase_beta_counts)

def kinopt_jax_prediction(theta):
    alpha = theta[:total_alpha]
    beta = theta[total_alpha:]
    # Kinase effects M[k, t]
    M_rows = []
    for k in range(len(unique_kinases)):
        start = int(kinase_beta_starts[k]); count = int(kinase_beta_counts[k])
        M_rows.append(jnp.sum(beta[start:start+count, None] * K_j[start:start+count, :], axis=0))
    M = jnp.stack(M_rows)
    preds = []
    for i in range(P_dense.shape[0]):
        start = int(gene_alpha_starts[i]); count = int(gene_kinase_counts[i])
        kin_idx = gene_kinase_idx_j[start:start+count]
        preds.append(jnp.sum(alpha[start:start+count, None] * M[kin_idx, :], axis=0))
    return jnp.clip(jnp.stack(preds), 0.0)

def kinopt_objective_jax(theta):
    pred = kinopt_jax_prediction(theta)
    return jnp.mean((P_j - pred) ** 2)

def kinopt_projection(theta, _):
    a = project_alpha_blocks(theta[:total_alpha], alpha_block_ids)
    b = project_beta_blocks(theta[total_alpha:], beta_block_ids, lower=0.0, upper=1.0)
    return jnp.concatenate([a, b])

solver = jaxopt.ProjectedGradient(kinopt_objective_jax, kinopt_projection, maxiter=8 if FAST_NOTEBOOK else 100, tol=1e-6, stepsize=0.0)

def make_start(seed):
    rng = np.random.default_rng(seed)
    raw = rng.uniform(0, 1, total_alpha + n_beta)
    return np.asarray(kinopt_projection(jnp.asarray(raw), None))

outcomes = []
for start_id, seed in enumerate([11, 12, 13] if FAST_NOTEBOOK else range(20)):
    theta0 = make_start(seed)
    params, state = solver.run(theta0, hyperparams_proj=None)
    params = np.asarray(kinopt_projection(params, None))
    package_loss = float(_objective(params, P_dense, t_max, P_dense.shape[0], gene_alpha_starts, gene_kinase_counts,
                                    gene_kinase_idx, total_alpha, kinase_beta_starts, kinase_beta_counts,
                                    K_data, K_indices, K_indptr, time_weights, 0))
    outcomes.append(SimpleNamespace(start_id=start_id, seed=seed, result=state, optimized_params=params,
                                    fun=package_loss, success=True, constr_violation=0.0, runtime_s=np.nan))

ranked = sorted(outcomes, key=lambda o: o.fun)
summary_df = pd.DataFrame([{"rank": r+1, "start_id": o.start_id, "seed": o.seed, "objective_loss": o.fun} for r, o in enumerate(ranked)])
display(summary_df)
best = ranked[0].optimized_params
print("Best start:", ranked[0].start_id, "loss:", ranked[0].fun)


## Outputs and interpretation

The table above is a **ranked multistart solution ensemble**. The first row is the best local solution among starts because it has the lowest scalar loss. We now convert fitted parameters and fitted curves into human-readable tables.

In [ ]:

alpha_values, beta_values = extract_parameters(P_initial, gene_kinase_counts, total_alpha, unique_kinases, K_index, best)
param_rows = []
for site, vals in alpha_values.items():
    for kinase, value in vals.items():
        param_rows.append({"parameter": "alpha", "substrate_site": str(site), "kinase": kinase, "value": value})
for (kinase, psite), value in beta_values.items():
    param_rows.append({"parameter": "beta", "substrate_site": psite, "kinase": kinase, "value": value})
params_df = pd.DataFrame(param_rows)
P_est, residuals, mse, rmse, mae, mape, r2 = compute_metrics(best, P_dense, t_max, gene_alpha_starts, gene_kinase_counts,
                                                            gene_kinase_idx, total_alpha, kinase_beta_starts,
                                                            kinase_beta_counts, K_data, K_indices, K_indptr)
fit_df = pd.DataFrame(P_est, index=[str(k) for k in P_initial.keys()], columns=time_cols)
metrics_df = pd.DataFrame([{"mse": mse, "rmse": rmse, "mae": mae, "mape": mape, "r_squared": r2}])
display(params_df)
display(metrics_df)
display(fit_df.iloc[:, :6])


## Visualization and saving

KinOpt includes a multistart summary plotting function; we use it for the ranked loss diagnostic. We also create compact fitted-vs-observed and parameter plots for notebook readability.

In [ ]:

output_dir = REPO_ROOT / "notebooks" / "outputs" / "kinopt"
output_dir.mkdir(parents=True, exist_ok=True)
params_df.to_csv(output_dir / "fitted_parameters.csv", index=False)
fit_df.to_csv(output_dir / "fitted_timeseries.csv")
metrics_df.to_csv(output_dir / "fit_metrics.csv", index=False)
export_outcomes_to_csv(ranked, output_dir / "multistart_summary.csv")
plot_multistart_summary_runtime_overlay(output_dir / "multistart_summary.csv", out_path=output_dir / "multistart_fun_vs_rank_runtime.png")

fig, ax = plt.subplots(figsize=(6, 3.5))
for i, label in enumerate(fit_df.index):
    ax.plot(t[:6], P_dense[i, :6], "o--", label=f"observed {label}")
    ax.plot(t[:6], P_est[i, :6], "-", label=f"fitted {label}")
ax.set_xlabel("time index")
ax.set_ylabel("phosphorylation signal")
ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig(output_dir / "fitted_vs_observed.png", dpi=200)
plt.show()

fig, ax = plt.subplots(figsize=(6, 3))
params_df.plot.bar(x="kinase", y="value", ax=ax, legend=False)
ax.set_ylabel("fitted weight")
fig.tight_layout()
fig.savefig(output_dir / "parameter_weights.png", dpi=200)
plt.show()
print("Saved files:", sorted(p.name for p in output_dir.iterdir()))


## End-to-end summary

- **Inputs:** substrate phosphorylation table plus kinase–substrate interaction table.
- **Preprocessing:** KinOpt converted biological labels into dense/sparse arrays and parameter blocks.
- **Model:** kinase effects were combined with \(\beta\), then assigned to substrates with \(\alpha\).
- **Solving:** a fast local JAXopt projected-gradient loop generated a ranked multistart solution ensemble and the KinOpt objective scored each solution.
- **Outputs:** parameter CSVs, fitted time series, fit metrics, and plots were saved under `notebooks/outputs/kinopt/`.
- **How KinOpt differs:** TFOpt uses a similar alpha/beta decomposition for TF regulation, whereas `phoskintime.protwise` and `networkmodel` fit ODE dynamics with Diffrax-based ODE solving.